# CALM-VAD on NWPU-Campus (Colab, full dataset)

Closes the one real remaining gap in the CALM-VAD paper. This is a
**different scale** from the other notebook: **76.6 GB** of video, **242**
real test clips, likely **hours** of GPU time -- so this notebook is built
to survive a Colab disconnect, not just run once.

**Before you start:**
1. `Runtime -> Change runtime type -> T4 GPU` (required, not optional here).
2. **Mount your Google Drive when Cell 2 asks** -- it is where the
   expensive part (per-clip pose-extraction results) is saved as it goes.
   Losing the ~76.6 GB of downloaded/extracted video to a disconnect just
   costs you a re-download (network time); losing GPU compute you already
   paid for would not.
3. Just run cells top to bottom, in order, once. **If Colab disconnects at
   any point, reconnect and re-run from Cell 1** -- every step below skips
   work it already finished (verified downloads, extracted videos, and
   especially finished pose-extractions are never redone).

**What this does, in order:** download all 106 archive volumes (MD5-checked,
auto-retries corrupt ones) -> extract with 7-Zip -> delete the archive to
reclaim disk -> convert the official ground-truth -> pose-extract all 242
real test clips on GPU (resumable, saved to Drive as it goes, each video
deleted right after its own extraction to keep disk use bounded) -> merge
-> run the real CALM-VAD harness -> print the exact numbers to paste back
into the paper.

No shortcuts, no subset: every real test clip, the same protocol as the
other four benchmarks (`\S` V in the paper).

In [ ]:
!pip -q install numpy scipy scikit-learn pyyaml gdown ultralytics opencv-python-headless
!apt-get -qq install -y p7zip-full > /dev/null

import os, sys, subprocess

def sh(cmd, check=False):
    """Run a shell command with output visible (never silently swallowed),
    so a failure here prints the real reason instead of a confusing error
    two lines later."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout)
    if r.stderr.strip():
        print(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')
    return r

REPO_URL = 'https://github.com/FaizanAbbas512/Sentrix.git'
for attempt in range(1, 4):
    sh('rm -rf /content/sentrix')
    r = sh('git clone --depth 1 %s /content/sentrix' % REPO_URL)
    if os.path.isdir('/content/sentrix/calm'):
        break
    print(f'clone attempt {attempt} did not produce /content/sentrix/calm -- retrying...')
else:
    raise RuntimeError('git clone failed 3 times -- check the error output above '
                        '(often a transient Colab network hiccup: Runtime -> '
                        'Restart session, then re-run this cell)')

os.chdir('/content/sentrix'); sys.path.insert(0, '/content/sentrix')
sh('python -m calm.selftest | tail -3')

import torch
print('torch version:', torch.__version__, '| cuda available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    # two different problems look identical here (torch.cuda.is_available()
    # is False) but need different fixes -- nvidia-smi tells them apart:
    hw = sh('nvidia-smi --query-gpu=name --format=csv,noheader')
    gpu_hw_present = hw.returncode == 0 and hw.stdout.strip()
    if gpu_hw_present:
        print(f'GPU hardware IS present ({hw.stdout.strip()}) but this torch build has no')
        print('CUDA support -- installing torch pulled in a CPU-only build from PyPI')
        print('(this happens because plain "pip install torch" defaults to CPU-only;')
        print('Colab pre-installs a CUDA build, but the earlier pip install line can')
        print('silently replace it). Reinstalling the correct CUDA build now...')
        sh('pip -q install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu124')
        print('Reinstalled. torch (and its CUDA bindings) cannot be reliably reloaded')
        print('into an already-running Python process -- this notebook needs one restart')
        print('now: Runtime -> Restart session, then re-run this cell (Cell 1) again.')
        print('The reinstalled torch will be picked up cleanly on the fresh process, and')
        print('this check will pass without reinstalling again (already correct).')
    else:
        print('No GPU hardware allocated to this runtime at all.')
        print('Fix: Runtime -> Disconnect and delete runtime, then Runtime -> Run all')
        print('(make sure Runtime -> Change runtime type -> T4 GPU is selected first).')

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print('\npose-extraction device:', DEVICE)
if DEVICE == 'cpu':
    print('\n!! Still no GPU -- see the diagnosis above for which fix applies.')
    print('   242 real clips on CPU would take days, not hours.')

## Mount Drive -- where the expensive (GPU) work is saved as it happens

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything expensive to redo (per-clip pose-extraction JSON parts, and the
# final merged/harness outputs) lives under here -- survives a disconnect.
PERSIST = '/content/drive/MyDrive/calm_vad_nwpu'
PARTS_DIR = f'{PERSIST}/pose_parts'      # one small json per clip, the resumable checkpoint
POSE_JSON = f'{PERSIST}/nwpucampus.json'  # final merged pose file
os.makedirs(PARTS_DIR, exist_ok=True)
print('resumable checkpoints ->', PARTS_DIR)

## Get the real dataset (all 106 archive volumes)

The official NWPU-Campus video release (from the dataset authors' own
Drive folder, linked at https://campusvad.github.io/) is a 106-part split
`.7z` archive with no per-clip boundaries -- there is no way to fetch a
subset, the full 76.6 GB must be obtained to extract even one clip
(verified while investigating this for the paper).

**Do this one manual step first (10 seconds, in your browser) -- it is
what makes the rest of this notebook reliable:**

1. Open <https://drive.google.com/drive/folders/1_EztmkNpTPyVb4lM0m4rLTXgXo_LzgF1>
   (the dataset authors' own shared folder) in the **same Google account**
   you mounted in Cell 2.
2. Click the folder name **NWPUCampusDataset_7zs** at the top to select it,
   then right-click -> **Organize -> Add shortcut to Drive** (older Drive
   UI: right-click -> **Add shortcut to Drive**) -> choose **My Drive**.
   This adds a *link*, not a copy -- it costs none of your own Drive
   storage quota.
3. Give it about a minute to actually sync into the mounted Drive, then
   come back here and run the next cell.

**Why this step, and why the next cell now *requires* it instead of
falling back to a direct download:** two real runs of this notebook showed
that anonymous public-link downloads (`gdown`) are not viable at this
scale -- Google Drive blocks the whole Colab IP as automated abuse after
about 30 rapid anonymous downloads, and every retry after that just burns
GPU session time for nothing. A Drive shortcut is read through your own
authenticated Google account instead, which isn't subject to that block,
and the next cell finds it automatically -- no code changes needed on your
side, and it fails fast with a clear message (not a long doomed retry
loop) if it can't find the shortcut.

In [ ]:
import gdown, hashlib, os, time, shutil

RAW = '/content/nwpu_raw'
os.makedirs(f'{RAW}/videos', exist_ok=True)
os.makedirs(f'{RAW}/groundtruth', exist_ok=True)

VOLUME_IDS = {
    'NWPUCampusDataset.7z.001': '1_jaF0dSAyFfCAOhpiEA9AdBtl1DAdDud',
    'NWPUCampusDataset.7z.002': '1-E1OMMNdzK1WnTA9DjYhz2YfUhx9FFva',
    'NWPUCampusDataset.7z.003': '1xtoOoKq0rcSmUYQWufwClWFPMts2ePIY',
    'NWPUCampusDataset.7z.004': '1kcX51ahwShukVEnognATxwJ-iUMOQaLx',
    'NWPUCampusDataset.7z.005': '1-UNYdxaAGrGqwuIyu4ibwiHsV3IbmCtj',
    'NWPUCampusDataset.7z.006': '19HhAkCtd-_yp5VlVcvWEuoyPZxr4VXWZ',
    'NWPUCampusDataset.7z.007': '1DVH85SJd_DrOz3YMAGwBC-1qexkJvxkZ',
    'NWPUCampusDataset.7z.008': '1H2buYOuJDhI1KMaqDaGhg9ICoDVTbdes',
    'NWPUCampusDataset.7z.009': '1mwtYZdvpqUfA4f9PR2x6CQdg1eJQT4O9',
    'NWPUCampusDataset.7z.010': '1auDfJD2LwYiPbuErE_yEYwWOZpsqQviR',
    'NWPUCampusDataset.7z.011': '175Q6rAObx0ZvRQVOQq3Tf-8ompDMxyVi',
    'NWPUCampusDataset.7z.012': '1MRQ4xTGzdaIeY6whFD9hrtlj_fZ1kat7',
    'NWPUCampusDataset.7z.013': '1eoRPmHHXD2Z2rCTrTdSSRf5cR7xa8Mt_',
    'NWPUCampusDataset.7z.014': '1z6Ot2M_FfmplQhSayx2XKOTfK38mhIwF',
    'NWPUCampusDataset.7z.015': '1zOQqqloFOplhISWObrsz-EYdBnEgJOjF',
    'NWPUCampusDataset.7z.016': '1NfWZFL5oLdfm5wniUndfu8QlNN6wlYMf',
    'NWPUCampusDataset.7z.017': '1yD99BAF-OrxzxOacVmURaawEyN6j9mdt',
    'NWPUCampusDataset.7z.018': '1Knid7c5hcQHixEiTmJHiTV6NrOHqHFR5',
    'NWPUCampusDataset.7z.019': '11D_OW5mao7j8AyGPXYi76-wxNNYWn9uI',
    'NWPUCampusDataset.7z.020': '1vypY63e6fwW_BqCcfI5hC0LYZ-dJ-SIs',
    'NWPUCampusDataset.7z.021': '1W9xhzlJUj0hUEPn8Z9-3KVmC48CzfGQl',
    'NWPUCampusDataset.7z.022': '1HRbUHxrwQr2-ahrRYYh5eCGUv7lgcds6',
    'NWPUCampusDataset.7z.023': '1VdFmE7xyNXNYRhnFvv4ihSLiUf1n9sKZ',
    'NWPUCampusDataset.7z.024': '1iuzDoLX1dAn_mxblXWdwOVxkSKCyPx_7',
    'NWPUCampusDataset.7z.025': '1stO5vN4n9eJw-64uK8x1ARj1sD7SeiNz',
    'NWPUCampusDataset.7z.026': '1a31Qc1lovvhw7LVjS0gAHJ0KLE9HNOEU',
    'NWPUCampusDataset.7z.027': '1p9QRWceKyBZQXg2CU68jWz52yX07smWJ',
    'NWPUCampusDataset.7z.028': '1MuQV7VPmEDs5rfHzFdTzOFUNN2ehlhwn',
    'NWPUCampusDataset.7z.029': '1nO42ouPHTPEzhIfoJpGKZ4Mv8S1BJEI5',
    'NWPUCampusDataset.7z.030': '1FiKsAMOCb0_Av0L0fPRjtUUz5BTcLXVD',
    'NWPUCampusDataset.7z.031': '1OZuGkubuJ-E81KQA9XVtF3jWLSDJzgPG',
    'NWPUCampusDataset.7z.032': '1q_mYFPkyOKNlUJCPgUq7UsxH73lQy6q2',
    'NWPUCampusDataset.7z.033': '1UmxIzYyZUxHxqpmp0PbG7-45y3klqO8y',
    'NWPUCampusDataset.7z.034': '1sPobfJdBdq3I798YzQ9-463k1tkGIjuO',
    'NWPUCampusDataset.7z.035': '1eJUfiRZBmLAyAjpiQMCl7eMNcvH4c2HX',
    'NWPUCampusDataset.7z.036': '1AEh44CTC5DW_mxInuDDdtL6NQx2f251t',
    'NWPUCampusDataset.7z.037': '12oTOd2XZndr9NjeCcCPOYtZnmpRnxZq0',
    'NWPUCampusDataset.7z.038': '1wL4vH1wLjiQ-Ds24quUY2cX1rhGoPRvF',
    'NWPUCampusDataset.7z.039': '1Gtlh98YAWSNDGI_JEmkalIxzTi9wRXRd',
    'NWPUCampusDataset.7z.040': '1W-BMlh7R6UVqD5qcnHf-2T0W_llTytA_',
    'NWPUCampusDataset.7z.041': '106Ji6uAMhVC5WNHeIZqxKzEA6QJKlMXf',
    'NWPUCampusDataset.7z.042': '1XMVEnl068-M4C3Gfuw08Verqg1X9LzQ6',
    'NWPUCampusDataset.7z.043': '1g59RVgafl2OhF0ly37D4kGGfrd7MUDSR',
    'NWPUCampusDataset.7z.044': '1x2P6b6XBzMyIIk0hxDbAakPTB6sWpBii',
    'NWPUCampusDataset.7z.045': '1iQrc7ruZU899N9D_gu2TeJVLDHQA775u',
    'NWPUCampusDataset.7z.046': '1QS2rVMTGfC1s_RQwfe_bG8SPG0rKNHlY',
    'NWPUCampusDataset.7z.047': '1cakOe9hJ3Nw96wI8rCsK16Zwl931ylPa',
    'NWPUCampusDataset.7z.048': '1l2gfkxmBeRTDdMydgajEB_oOK7-sPAmT',
    'NWPUCampusDataset.7z.049': '1mZdv0rY68e0h8PpuDskxeKdrMqe5Jfz7',
    'NWPUCampusDataset.7z.050': '1TvqVIkWhYCPUHvw570psr8cMqQ60M4zm',
    'NWPUCampusDataset.7z.051': '1klOsw9dSd2PTYPXUHX1p2STsin6vZe1c',
    'NWPUCampusDataset.7z.052': '1zXnJhcRu1iTAwdJ9m9DIUMwzI7kmaQ1V',
    'NWPUCampusDataset.7z.053': '1--tVKfpG3zJJMgjRWj4IP5UOL3EKddjg',
    'NWPUCampusDataset.7z.054': '1tt4al4Y9MA--DoGOD8cFS9Udv2Op0orr',
    'NWPUCampusDataset.7z.055': '1WwldN7TFksPBXZvNrRfuqAJGM1_tianV',
    'NWPUCampusDataset.7z.056': '15z8RxgT2lFZUGm4iVjE0AC6WtTWU5UP6',
    'NWPUCampusDataset.7z.057': '1FxMf0Nv8HJzule9xJhsvsiwjX07IL3L4',
    'NWPUCampusDataset.7z.058': '1KxVa8ymB8m3I84TVvMpqnOHVQXjwL0ob',
    'NWPUCampusDataset.7z.059': '1252fb2-2vspjI-a9mVf6YLLrLdCRiaYZ',
    'NWPUCampusDataset.7z.060': '19Ut3849EFWpK65PpDlMwE_DCRW056on9',
    'NWPUCampusDataset.7z.061': '1KTtKFHSAKllYshrE_Ant3GHmx2-2vLQI',
    'NWPUCampusDataset.7z.062': '1yP7kYiK6pqweePwHLIcoHjROENa519B_',
    'NWPUCampusDataset.7z.063': '1Xka9Q0kOfFMrCWD3NvAJ4G62XpRs78KD',
    'NWPUCampusDataset.7z.064': '1wyDsFU8GipnfLfmStRqMEE73KRkFNt_A',
    'NWPUCampusDataset.7z.065': '1sk92Le-k-ZhocvN6FBsh4z7ms9aeW-Sw',
    'NWPUCampusDataset.7z.066': '1184fPjYgDJWX81fuZgItdBtYvbU19wsT',
    'NWPUCampusDataset.7z.067': '1nsXycvhH2_ghruW8Cy6x_gzxNJX95WBn',
    'NWPUCampusDataset.7z.068': '1NCTVh-kHTbKqecycOdJT87AGfinogKHg',
    'NWPUCampusDataset.7z.069': '11xRu15U72Sd2TUWeKgZsmmFd3NQ7TFSQ',
    'NWPUCampusDataset.7z.070': '1BbIJqTKvPeoafswqGYgijYC9RxTQueS3',
    'NWPUCampusDataset.7z.071': '1ZnYOGzP0ZGjCF37FDFctBCQdT0BUOTXx',
    'NWPUCampusDataset.7z.072': '1XdPi1xF14G3LjLDyuQkGifaaEMbboB3D',
    'NWPUCampusDataset.7z.073': '1JvG1jJm2ZvL5n3hXHjcH3bSXfIR-Nsg_',
    'NWPUCampusDataset.7z.074': '1J5xRGGDZqX142uPGuMD8efmBgppRSieW',
    'NWPUCampusDataset.7z.075': '14LQEgY5XsnMni9m-ezzLrdrjVZh3f49U',
    'NWPUCampusDataset.7z.076': '1wWuatXWNo2XvhUoWu9n2iz1JJQzxSKzI',
    'NWPUCampusDataset.7z.077': '144IY2lefntL0SotmhSmtLUVFZMFsGOP2',
    'NWPUCampusDataset.7z.078': '1vNzxsvzV9lNn_ZXS_yLjQ8HkdGr9olF0',
    'NWPUCampusDataset.7z.079': '10WU1LOsu55d1aUV4UyHtT_zaex11u9F_',
    'NWPUCampusDataset.7z.080': '1Qh6NgForXHqV0vDSKp3tT88SEGAP_C1o',
    'NWPUCampusDataset.7z.081': '1t8Hk7b5MnxU6Whj80tyl1qqvXP9AggAV',
    'NWPUCampusDataset.7z.082': '1YIW5SYpDfQkt7zqx8YyQMbn4kckGV_vW',
    'NWPUCampusDataset.7z.083': '15jaIptB9_2zyO5psF824j2c4SK3mK7k7',
    'NWPUCampusDataset.7z.084': '1H5Gd_EJxEFU3g9lUtmoXDLytd1R7v9cp',
    'NWPUCampusDataset.7z.085': '1pvn0w1GjmEmG_Me3umphFYKY1zptYZuy',
    'NWPUCampusDataset.7z.086': '18oiInKucRyG0Z_CdnGfMoMmx9X6mQbyW',
    'NWPUCampusDataset.7z.087': '12hVrLUwa0XOU4d-j49LCHQakwB6lr4sG',
    'NWPUCampusDataset.7z.088': '17vxNAmYKFxUnz652hFtARe1xajiomugG',
    'NWPUCampusDataset.7z.089': '1JMGJZ_SCAjvMmr4Z7vEQTsP_AsVr0499',
    'NWPUCampusDataset.7z.090': '1eJ-2iDCOfKJpuGWXMdAZH7skYASp2Lkp',
    'NWPUCampusDataset.7z.091': '1oR28KPOTQC3SsTnZ45NwJYHbU7d9Fh7C',
    'NWPUCampusDataset.7z.092': '1YCj9BgvkdqVAk2YmGkSuGHnME5i5f57H',
    'NWPUCampusDataset.7z.093': '19L75PKxUSCYnZ6_P7tFHmdwVLxTbLqAl',
    'NWPUCampusDataset.7z.094': '1FVTXB3nnS2vniPOJwhX9GoazJKkt8oD7',
    'NWPUCampusDataset.7z.095': '1Hm1CXkLur5l9M94-F8JC38-bGQ0imY0M',
    'NWPUCampusDataset.7z.096': '1breRDDZTa1nGCSY4w1cdrXl6uocAOKqw',
    'NWPUCampusDataset.7z.097': '1IiGH5V23UYkGMk_RY2hTUDHzGGjP9SeG',
    'NWPUCampusDataset.7z.098': '1DsxG0INcSlxlVtsQN0Hl3gbzW9ddxgXx',
    'NWPUCampusDataset.7z.099': '1HH-3xW25zPz6phVcha9z8zY0yJ5W5TkU',
    'NWPUCampusDataset.7z.100': '1z96CYwRd3gmHVJmhUByDGsqL4_fNadvH',
    'NWPUCampusDataset.7z.101': '1WiNOjpO8KffVQGKABkhp0T9CokrfcB_R',
    'NWPUCampusDataset.7z.102': '1iY5W_LEJFAhT_RPSSprRtkBd3_mWWHfg',
    'NWPUCampusDataset.7z.103': '1-weGVll6656y88yrWbjQSYcSW4GYTmbv',
    'NWPUCampusDataset.7z.104': '1s6eLOGWKnTHzC0GOKIGB6Q0BD7NtZwv5',
    'NWPUCampusDataset.7z.105': '1sEzVanrY6ecvftqw1gUjijM618p5B1Da',
    'NWPUCampusDataset.7z.106': '1s_RXa5nr2Xij7wq4CQ6XQYCZ1NSyhc9d',
}
GT_NPZ_ID = '1-gLI2cMZ_PEkWM0uIdQkjrB_y8Fwp-oL'
MD5_ID    = '1Q7oiEhgDJhNcb79NIRbmRDebg8ENssVG'
VOLUME_NAMES = sorted(VOLUME_IDS)

def _md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

# --- find a Drive shortcut to the dataset folder, FAST (no slow full-tree walk:
# Drive's FUSE mount in Colab is slow per directory listing, and a deep os.walk
# across a Drive account with other large folders can take a very long time or
# never finish before you'd give up waiting) ---
def find_drive_shortcut(timeout_s=25):
    root = '/content/drive/MyDrive'
    if not os.path.isdir(root):
        return None, 'root'

    # 1) exact expected path -- instant, covers the normal case (shortcut
    #    added at the top level of My Drive, original folder name kept)
    direct = f'{root}/NWPUCampusDataset_7zs/videos'
    if os.path.isfile(f'{direct}/NWPUCampusDataset.7z.001'):
        return direct, 'direct'

    # 2) shallow scan of MyDrive's immediate children only (one listdir call,
    #    not recursive) for anything with 'nwpu' or 'campus' in the name --
    #    covers a renamed shortcut or one dropped in a slightly different spot
    try:
        top_level = os.listdir(root)
    except Exception as e:
        return None, f'listdir failed: {e}'
    candidates = [d for d in top_level if 'nwpu' in d.lower() or 'campus' in d.lower()]
    for name in candidates:
        for sub in ('videos', '.'):
            p = f'{root}/{name}/{sub}'
            if os.path.isfile(f'{p}/NWPUCampusDataset.7z.001'):
                return p, 'shallow'

    # 3) last resort: a TIME-BOUNDED shallow walk (max 2 levels deep, gives up
    #    after timeout_s rather than hanging) -- reports what it saw either way
    start = time.time()
    for dirpath, dirnames, filenames in os.walk(root):
        if time.time() - start > timeout_s:
            return None, f'gave up after {timeout_s}s scanning (Drive mount is slow); ' \
                          f'top-level MyDrive contents: {top_level}'
        if 'NWPUCampusDataset.7z.001' in filenames:
            return dirpath, 'walk'
        if dirpath.count(os.sep) - root.count(os.sep) >= 2:
            dirnames[:] = []
    return None, f'not found; top-level MyDrive contents: {top_level}'

drive_videos_dir, how = find_drive_shortcut()
if drive_videos_dir:
    print(f'Found the Drive shortcut at: {drive_videos_dir}  (via: {how})')
    print('Reading volumes through your authenticated Drive session (reliable, no quota).\n')
else:
    print('STOPPING: no Drive shortcut found. Diagnostic:', how)
    print()
    print('This step deliberately does NOT fall back to anonymous downloads any more --')
    print('two real runs of this notebook showed that path always fails at scale (Google')
    print('blocks the whole Colab IP after ~30 files) and just burns your GPU session time')
    print('waiting through 106 x 3 doomed retries for nothing.')
    print()
    print('Fix: go add the shortcut (Cell 4 instructions), giving it a minute to sync, then')
    print('just re-run this cell alone (no need to redo Cells 1-3). If you DID already add')
    print('it and this still fails, check the "top-level MyDrive contents" list printed')
    print('above -- if NWPUCampusDataset_7zs really isn\'t in it, the shortcut may have been')
    print('added to a different Google account than the one Colab mounted in Cell 3.')
    raise SystemExit('no Drive shortcut -- see message above')

# official checksums, so a corrupt/partial copy is caught and retried
# automatically rather than silently poisoning the archive
md5_path = f'{RAW}/md5.txt'
if not os.path.exists(md5_path) and os.path.exists(f'{drive_videos_dir}/../md5.txt'):
    shutil.copy(f'{drive_videos_dir}/../md5.txt', md5_path)
official_md5 = dict(line.split() for line in open(md5_path)) if os.path.exists(md5_path) else {}

def get_verified(name, dest, expected_md5=None):
    """Copy+verify one volume from the Drive shortcut. Never raises."""
    if os.path.exists(dest):
        if expected_md5 is None or _md5(dest) == expected_md5:
            return True
        os.remove(dest)
    src = f'{drive_videos_dir}/{name}'
    if not os.path.exists(src):
        print(f'  {name}: not found under the Drive shortcut')
        return False
    try:
        shutil.copy(src, dest)
    except Exception as e:
        print(f'  {name}: copy error: {type(e).__name__}: {e}')
        if os.path.exists(dest):
            os.remove(dest)
        return False
    return expected_md5 is None or (os.path.exists(dest) and _md5(dest) == expected_md5)

print(f'Getting {len(VOLUME_NAMES)} volumes from the Drive shortcut (safe to re-run --')
print('already-verified ones are skipped instantly)...')
failed = []
for i, name in enumerate(VOLUME_NAMES):
    dest = f'{RAW}/videos/{name}'
    good = get_verified(name, dest, official_md5.get(name))
    if not good:
        failed.append(name)
    print(f'  [{i+1}/{len(VOLUME_NAMES)}] {name}  {"OK" if good else "FAILED"}')

ok = len(VOLUME_NAMES) - len(failed)
print(f'\n{ok}/{len(VOLUME_NAMES)} volumes ready.')
if failed:
    print(f'\n{len(failed)} volume(s) missing under the Drive shortcut itself -- the shortcut')
    print('may point at an incomplete copy, or Drive is still syncing. Wait a minute and')
    print('re-run this cell (already-good volumes are skipped, only these are retried).')
else:
    gt_dest = f'{RAW}/groundtruth/NWPU_Campus_gt.npz'
    gt_src = f'{drive_videos_dir}/../groundtruth/NWPU_Campus_gt.npz'
    if os.path.exists(gt_src):
        shutil.copy(gt_src, gt_dest)
    print('ground truth ready. All volumes ready -- continue to the next cell.')

## Extract (all 106 volumes at once, then delete them to reclaim ~76.6 GB)

In [ ]:
EXTRACT_DIR = '/content/nwpu_videos'
os.makedirs(EXTRACT_DIR, exist_ok=True)

present = [n for n in VOLUME_IDS if os.path.exists(f'{RAW}/videos/{n}')]
assert len(present) == len(VOLUME_IDS), (
    f'only {len(present)}/{len(VOLUME_IDS)} volumes are on disk -- go back to the '
    'download cell above and re-run it until it reports 0 failed volumes, then come back here')

marker = f'{EXTRACT_DIR}/.extracted_ok'
if os.path.exists(marker):
    print('already extracted (marker present) -- skipping extraction.')
else:
    # 7z auto-detects the multi-volume set from .7z.001; -aos = skip files
    # that already extracted correctly, so a re-run after a disconnect
    # resumes instead of restarting
    rc = subprocess.run(
        ['7z', 'x', f'{RAW}/videos/NWPUCampusDataset.7z.001',
         f'-o{EXTRACT_DIR}', '-aos', '-y'],
    ).returncode
    n_files = sum(len(fs) for _, _, fs in os.walk(EXTRACT_DIR))
    print(f'extraction rc={rc}, {n_files} files now in {EXTRACT_DIR}')
    assert rc == 0 and n_files > 500, 'extraction looks incomplete -- re-run this cell'
    open(marker, 'w').close()

    # archive volumes are no longer needed once extraction succeeded --
    # delete them now to reclaim ~76.6 GB before we need disk for videos
    sh(f'rm -rf {RAW}/videos')
    print('deleted the 106 archive volumes (extraction verified OK).')

## Ground truth -> per-clip files, and find the 242 real test videos

In [ ]:
GT_NPY_DIR = '/content/nwpu_gt_npy'
sh(f'python -m calm.nwpu_gt --npz {RAW}/groundtruth/NWPU_Campus_gt.npz --out {GT_NPY_DIR}')

import numpy as np
test_keys = sorted(f[:-4] for f in os.listdir(GT_NPY_DIR) if f.endswith('.npy'))
print(f'{len(test_keys)} real test clips in the official ground truth')
assert len(test_keys) == 242, f'expected 242 test clips, found {len(test_keys)} -- check the download'

# match GT keys to the actual extracted video files (whatever their real
# extension/subfolder layout turns out to be -- don't assume, search)
video_by_key = {}
for root, _, files in os.walk(EXTRACT_DIR):
    for fn in files:
        stem, ext = os.path.splitext(fn)
        if stem in test_keys and ext.lower() in ('.avi', '.mp4', '.mkv', '.mov'):
            video_by_key[stem] = os.path.join(root, fn)

missing = [k for k in test_keys if k not in video_by_key]
print(f'{len(video_by_key)}/{len(test_keys)} test videos located on disk')
if missing:
    print('missing (first 10):', missing[:10])
assert not missing, 'some test videos were not found after extraction -- check EXTRACT_DIR layout above'

## Pose-extract all 242 real test clips (GPU, resumable)

This is the long step. Each clip's YOLO11n-pose + ByteTrack result is saved
to your Drive (`PARTS_DIR`) the moment it finishes, and its raw video is
deleted right after to keep local disk bounded. **If this disconnects,
just re-run this cell** -- every clip whose part file already exists in
Drive is skipped instantly, so you only ever redo the clip that was
in-flight when it disconnected, never the ones already done.

In [ ]:
from calm.extract_poses import extract_one, _load_gt
from ultralytics import YOLO
import cv2, json, time

model = YOLO('yolo11n-pose.pt')
classes = [0, 24, 26, 28]   # person + backpack/handbag/suitcase (abandoned-object cue)

done = sorted(f[:-5] for f in os.listdir(PARTS_DIR) if f.endswith('.json'))
todo = [k for k in test_keys if k not in done]
print(f'{len(done)} clips already done (in Drive), {len(todo)} left to process.')

t0 = time.time()
for i, key in enumerate(todo):
    vp = video_by_key[key]
    part_path = f'{PARTS_DIR}/{key}.json'
    cap = cv2.VideoCapture(vp)
    vfps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()

    n, frames = extract_one(model, vp, classes, imgsz=640, conf=0.35, iou=0.5,
                             tracker='bytetrack.yaml', stride=1, device=DEVICE)
    clip = {'name': key, 'n_frames': n, 'split': 'test',
            'gt': _load_gt(GT_NPY_DIR, key, n), 'frames': frames}
    with open(part_path, 'w', encoding='utf-8') as f:
        json.dump(clip, f)

    os.remove(vp)   # free local disk immediately; the part in Drive is the source of truth now

    elapsed = time.time() - t0
    rate = (i + 1) / elapsed if elapsed > 0 else 0
    eta_min = (len(todo) - i - 1) / rate / 60 if rate > 0 else float('inf')
    print(f'  [{i+1}/{len(todo)}] {key}  ({n} frames)  '
          f'elapsed={elapsed/60:.1f}min  eta={eta_min:.0f}min')

print('\nAll clips processed. Re-run this cell any time -- it will report 0 left to process.')

## Merge the 242 parts into the final pose file, and run the real harness

In [ ]:
import numpy as np

done = sorted(f[:-5] for f in os.listdir(PARTS_DIR) if f.endswith('.json'))
assert len(done) == 242, f'only {len(done)}/242 parts ready -- re-run the extraction cell above to finish'

fps_seen = []
clips = []
for key in done:
    with open(f'{PARTS_DIR}/{key}.json', encoding='utf-8') as f:
        c = json.load(f)
    clips.append(c)

merged = {'fps': 25.0, 'clips': clips}   # NWPU-Campus is 25fps per the official release; harness re-derives event timing from this
os.makedirs('data/pose', exist_ok=True)
with open('data/pose/nwpucampus.json', 'w', encoding='utf-8') as f:
    json.dump(merged, f)
sh(f'cp data/pose/nwpucampus.json {POSE_JSON}')   # keep a copy in Drive too

n_gt = sum(len(c['gt']) for c in clips)
print(f'merged {len(clips)} real test clips, {n_gt} real ground-truth anomaly events -> data/pose/nwpucampus.json')

print('\nRunning the real CALM-VAD harness (same code, same metrics as the other 4 benchmarks)...')
sh('python -m calm.harness --generic data/pose/nwpucampus.json --tag nwpucampus')
print(open('results/calm_report_nwpucampus.txt').read())

## Done -- get the real numbers back

- `results/calm_report_nwpucampus.txt` / `.json` -- the real numbers, printed above too.
- Download both, plus `data/pose/nwpucampus.json` (also saved to your Drive
  at `PERSIST`), and send them back -- they get pasted directly into
  `paper/main.tex`'s cross-benchmark table and the NWPU `\pend{}` markers
  get removed, using these real numbers, the same way the other four
  benchmarks were done. No numbers in this notebook are synthetic or
  estimated -- every one comes from this run, on the real, official
  NWPU-Campus test set.

In [ ]:
from google.colab import files
files.download('results/calm_report_nwpucampus.json')
files.download('results/calm_report_nwpucampus.txt')
files.download('data/pose/nwpucampus.json')